# M05-P01 — Tokens, contexto y presupuesto de generación

En M01 ya vimos temperatura, límites de salida, contexto y streaming. Aquí no repetimos aquello.

La pregunta ahora es:

```text
¿qué entra realmente en el contexto del modelo
y cómo puedo medirlo antes de inferencia?
```

Usaremos Amazon Bedrock `CountTokens` con el mismo modelo del curso.

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt

from enterprise_genai.config import get_settings
from enterprise_genai.models.bedrock import bedrock_runtime

settings = get_settings()
runtime = bedrock_runtime()
MODEL_ID = settings.bedrock_text_model_id

CONTEXT_WINDOW = 200_000
MAX_OUTPUT_TOKENS = 64_000

print('Modelo:', MODEL_ID)
print('Región:', settings.aws_region)
print('Ventana usada en el ejercicio:', f'{CONTEXT_WINDOW:,}', 'tokens')

## 1. Contar tokens antes de llamar al modelo

`CountTokens` recibe la misma estructura conversacional que `Converse`. Así podemos medir mensajes e instrucciones antes de generar una respuesta.

In [ ]:
def user_message(text):
    return [{'role': 'user', 'content': [{'text': text}]}]

def count_converse_tokens(messages, system_text=None):
    converse_input = {'messages': messages}
    if system_text:
        converse_input['system'] = [{'text': system_text}]

    response = runtime.count_tokens(
        modelId=MODEL_ID,
        input={'converse': converse_input},
    )
    return response['inputTokens']

print('Tokens para una petición con «Hola»:', count_converse_tokens(user_message('Hola')))

## 2. Palabras, caracteres y tokens no son lo mismo

`split()` solo cuenta palabras: no es un tokenizer. Además, `CountTokens` mide la petición conversacional completa, incluido su overhead.

In [ ]:
examples = {
    'frase corta': 'La VPN no funciona.',
    'frase técnica': 'La autenticación SAML de la VPN devuelve HTTP 401 desde las 08:15.',
    'identificador': 'INC-2026-10-18-ACME-VPN-004821',
    'emoji': 'VPN caída 🔐🚨 — revisar autenticación',
    'json': json.dumps({'service': 'vpn', 'status': 'degraded', 'users': 73}),
    'python': 'if status_code == 401:\n    retry_authentication(user_id)',
}

rows = []
for label, text in examples.items():
    tokens = count_converse_tokens(user_message(text))
    rows.append({
        'ejemplo': label,
        'caracteres': len(text),
        'palabras': len(text.split()),
        'input_tokens': tokens,
        'chars/token': round(len(text) / tokens, 2),
    })

token_df = pd.DataFrame(rows)
token_df

## 3. El formato y el `system prompt` también consumen contexto

Comparamos información equivalente con formatos distintos y medimos el efecto de las instrucciones de sistema.

In [ ]:
variants = {
    'conciso': '73 usuarios no pueden autenticarse en la VPN desde las 08:15.',
    'verboso': (
        'Se ha detectado una situación en la que un total de 73 usuarios de ACME '
        'no consiguen completar correctamente el proceso de autenticación requerido '
        'para acceder al servicio corporativo de VPN desde aproximadamente las 08:15.'
    ),
    'json': json.dumps({
        'affected_users': 73,
        'service': 'VPN',
        'problem': 'authentication failure',
        'since': '08:15',
    }),
    'lista': '- usuarios: 73\n- servicio: VPN\n- problema: autenticación\n- inicio: 08:15',
}

variant_rows = []
for label, text in variants.items():
    variant_rows.append({
        'formato': label,
        'caracteres': len(text),
        'palabras': len(text.split()),
        'input_tokens': count_converse_tokens(user_message(text)),
    })

display(pd.DataFrame(variant_rows).sort_values('input_tokens'))

QUESTION = 'Resume el incidente en una frase.'
SYSTEM_SHORT = 'Eres un analista de operaciones.'
SYSTEM_LONG = (
    'Eres un analista senior de operaciones empresariales. Distingue hechos observados de hipótesis, '
    'evita afirmar una causa raíz sin evidencia, señala incertidumbres y prioriza acciones de validación.'
)

system_rows = []
for label, system_text in [
    ('sin system', None),
    ('system corto', SYSTEM_SHORT),
    ('system detallado', SYSTEM_LONG),
]:
    system_rows.append({
        'configuración': label,
        'input_tokens': count_converse_tokens(user_message(QUESTION), system_text),
    })

display(pd.DataFrame(system_rows))

## 4. Una conversación crece turno a turno

Para mantener contexto, una aplicación suele reenviar el historial relevante. Medimos cómo aumenta el número de tokens.

In [ ]:
conversation = [
    {'role': 'user', 'content': [{'text': 'Tenemos 73 usuarios sin acceso VPN desde las 08:15.'}]},
    {'role': 'assistant', 'content': [{'text': '¿Está afectado también el acceso web corporativo?'}]},
    {'role': 'user', 'content': [{'text': 'No. La web corporativa funciona correctamente.'}]},
    {'role': 'assistant', 'content': [{'text': '¿Qué muestra el estado del backend de autenticación?'}]},
    {'role': 'user', 'content': [{'text': 'El dashboard marca VPN AUTH BACKEND como DEGRADED.'}]},
    {'role': 'assistant', 'content': [{'text': 'Eso aporta correlación, pero todavía no confirma la causa raíz.'}]},
    {'role': 'user', 'content': [{'text': '¿Qué comprobarías a continuación?'}]},
]

history_rows = []
for end in [1, 3, 5, 7]:
    prefix = conversation[:end]
    history_rows.append({
        'mensajes': len(prefix),
        'input_tokens': count_converse_tokens(prefix, SYSTEM_SHORT),
    })

history_df = pd.DataFrame(history_rows)
display(history_df)

plt.figure(figsize=(7, 4))
plt.plot(history_df['mensajes'], history_df['input_tokens'], marker='o')
plt.xlabel('Mensajes incluidos')
plt.ylabel('Input tokens')
plt.title('Crecimiento del contexto con el historial')
plt.grid(alpha=0.25)
plt.show()

## 5. Predicción antes de inferencia vs uso real

Contamos tokens, ejecutamos exactamente esa conversación y comparamos con `usage.inputTokens`.

In [ ]:
INCIDENT = (
    'Desde las 08:15, 73 usuarios de ACME no pueden autenticarse en la VPN. '
    'El servicio web corporativo funciona correctamente. El equipo de operaciones está investigando.'
)
SYSTEM = 'Eres un analista de operaciones. Resume solo hechos disponibles y no inventes una causa raíz.'
messages = user_message(INCIDENT + '\n\nDevuelve un resumen de máximo tres viñetas.')

predicted_tokens = count_converse_tokens(messages, SYSTEM)
response = runtime.converse(
    modelId=MODEL_ID,
    system=[{'text': SYSTEM}],
    messages=messages,
    inferenceConfig={'temperature': 0, 'maxTokens': 180},
)

print(response['output']['message']['content'][0]['text'])
print()
print('CountTokens antes de inferencia:', predicted_tokens)
print('usage.inputTokens:', response['usage']['inputTokens'])
print('usage.outputTokens:', response['usage']['outputTokens'])
print('usage.totalTokens:', response['usage']['totalTokens'])
print('stopReason:', response['stopReason'])

## 6. Presupuesto de contexto y puente hacia RAG

Podemos razonar como una aplicación:

```text
ventana total - entrada - reserva de salida = margen disponible
```

Que un documento quepa no significa que enviarlo entero sea siempre la mejor estrategia: también importan relevancia, coste y latencia.

In [ ]:
def context_budget(input_tokens, reserve_output=2_000, context_window=CONTEXT_WINDOW):
    available = context_window - input_tokens - reserve_output
    return {
        'context_window': context_window,
        'input_tokens': input_tokens,
        'reserved_output_tokens': reserve_output,
        'available_tokens': max(0, available),
        'used_percent_before_output': round(100 * (input_tokens + reserve_output) / context_window, 3),
        'fits': available >= 0,
    }

print('Presupuesto de la petición anterior:')
print(context_budget(predicted_tokens))

paragraph = (
    'Procedimiento de acceso remoto. Los usuarios deben utilizar MFA para conectarse a la VPN. '
    'Los errores de autenticación deben correlacionarse con los logs del proveedor de identidad '
    'y con el estado del backend antes de declarar una causa raíz.'
)

long_rows = []
for n in [1, 10, 50, 100, 250]:
    text = '\n\n'.join([paragraph] * n)
    tokens = count_converse_tokens(user_message(text))
    long_rows.append({
        'bloques': n,
        'caracteres': len(text),
        'input_tokens': tokens,
        '% ventana': round(tokens / CONTEXT_WINDOW * 100, 3),
    })

display(pd.DataFrame(long_rows))

## 7. MODIFICA

Prueba una opción:

- añade dos turnos a la conversación;
- amplía el `system prompt`;
- añade una variante XML;
- reserva 8.000 tokens de salida;
- aumenta el documento hasta superar el 25 % de la ventana.

### Pregunta final

Si un documento completo cabe en contexto, ¿por qué podríamos preferir recuperar solo los fragmentos relevantes?

Guarda la pregunta: será central en embeddings y RAG.